## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-09-23-51-22 +0000,bbc,What we know so far about Jensen Huang's affai...,https://www.bbc.co.uk/news/articles/ckgel338xq...
1,2026-11-09-23-49-14 +0000,wapo,Supreme Court allows major part of Trump’s bor...,https://www.washingtonpost.com/politics/2026/1...
2,2026-11-09-23-48-34 +0000,nyt,Democrats Demand Answers on Trump’s A.I. Polic...,https://www.nytimes.com/2026/11/09/us/politics...
3,2026-11-09-23-47-28 +0000,nyt,Congress Has Reined In the Presidency Before. ...,https://www.nytimes.com/2026/11/09/magazine/co...
4,2026-11-09-23-47-23 +0000,cbc,Nvidia CEO Jensen Huang is embroiled in a spyi...,https://www.cbc.ca/news/business/nvidia-surviv...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2702/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
30,nvidia,48
13,trump,45
3,huang,31
7,spy,26
2,jensen,24


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
63,2026-11-09-22-16-39 +0000,nypost,Nvidia boss Jensen Huang had ‘absolutely no id...,https://nypost.com/2026/11/09/us-news/nvidia-b...,244
131,2026-11-09-20-39-13 +0000,nypost,Trader warned of Nvidia CEO Jensen Huang’s ‘un...,https://nypost.com/2026/11/09/business/trader-...,227
25,2026-11-09-23-10-47 +0000,wapo,Trump says Nvidia CEO denied knowing woman he ...,https://www.washingtonpost.com/technology/2026...,224
158,2026-11-09-19-47-46 +0000,nypost,Nvidia CEO Jensen Huang has no obvious success...,https://nypost.com/2026/11/09/business/nvidia-...,219
257,2026-11-09-14-57-09 +0000,nypost,Nvidia stock plunges 4% after CEO Jensen Huang...,https://nypost.com/2026/11/09/business/nvidia-...,212


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
63,244,2026-11-09-22-16-39 +0000,nypost,Nvidia boss Jensen Huang had ‘absolutely no id...,https://nypost.com/2026/11/09/us-news/nvidia-b...
395,113,2026-11-09-00-30-00 +0000,wsj,President Trump said Sunday that he was joking...,https://www.wsj.com/politics/policy/trump-says...
382,75,2026-11-09-02-00-00 +0000,wsj,Voters fed up with high prices gave Democrats ...,https://www.wsj.com/politics/elections/democra...
76,73,2026-11-09-22-06-00 +0000,wsj,Veteran GOP lawmaker Steve Scalise is going he...,https://www.wsj.com/politics/why-steve-scalise...
231,68,2026-11-09-16-38-04 +0000,chitri,Nvidia’s investigation into CEO Jensen Huang’s...,https://www.chicagotribune.com/2026/11/09/nvid...
160,64,2026-11-09-19-38-13 +0000,nypost,Rep. Thomas Massie reintroduces impeachment ar...,https://nypost.com/2026/11/09/us-news/rep-thom...
120,54,2026-11-09-21-11-22 +0000,chitri,Supreme Court clears the way for border wall c...,https://www.chicagotribune.com/2026/11/09/supr...
26,44,2026-11-09-23-08-41 +0000,nypost,Ex-cop accused of killing jogger went to work ...,https://nypost.com/2026/11/09/us-news/ex-cop-a...
304,43,2026-11-09-11-48-00 +0000,wsj,This week will pose two major tests for Presid...,https://www.wsj.com/politics/policy/nvidias-je...
266,43,2026-11-09-14-28-31 +0000,nypost,Russell Brand sentenced to over 10 years in pr...,https://nypost.com/2026/11/09/world-news/russe...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
